In [9]:
import polars as pl
import pandas as pd
from sklearn.ensemble import IsolationForest

In [10]:
# Load your clean, checkpointed data
print("Loading data...")
df = pl.read_parquet('../data/processed/train_data_imputed.parquet')

Loading data...


In [11]:
print("--- 1 & 2. Executing Safe Winsorization & Robust Scaling ---")

# Identify continuous numerical columns (exclude targets, dates, and binary/categorical encoded as int)
num_cols = [c for c in df.columns if df.schema[c] in [pl.Float32, pl.Float64] and c not in ['target', 'S_2']]

# We need to calculate the true bounds (ignoring -999.0)
exprs_bounds = []
for c in num_cols:
    valid_data = pl.when(pl.col(c) == -999.0).then(None).otherwise(pl.col(c))
    exprs_bounds.extend([
        valid_data.quantile(0.01).alias(f"{c}_p01"),
        valid_data.quantile(0.99).alias(f"{c}_p99"),
        valid_data.median().alias(f"{c}_median"),
        (valid_data.quantile(0.75) - valid_data.quantile(0.25)).alias(f"{c}_iqr")
    ])

# Calculate all bounds in one lightning-fast pass
bounds = df.select(exprs_bounds).to_dicts()[0]

# Now apply Winsorization and Robust Scaling, protecting -999.0
exprs_scale = []
for c in num_cols:
    p01 = bounds[f"{c}_p01"]
    p99 = bounds[f"{c}_p99"]
    median = bounds[f"{c}_median"]
    iqr = bounds[f"{c}_iqr"] if bounds[f"{c}_iqr"] > 0 else 1.0 # Prevent division by zero
    
    # 1. Clip (Winsorize) the real data
    winsorized = pl.col(c).clip(lower_bound=p01, upper_bound=p99)
    # 2. Scale the real data
    scaled = (winsorized - median) / iqr
    
    # 3. Apply conditional mask: If -999.0, keep -999.0, else apply scaled value
    exprs_scale.append(
        pl.when(pl.col(c) == -999.0).then(-999.0).otherwise(scaled).alias(c)
    )

df = df.with_columns(exprs_scale)
print(" Winsorization and Robust Scaling Complete.")

--- 1 & 2. Executing Safe Winsorization & Robust Scaling ---
 Winsorization and Robust Scaling Complete.


In [12]:
# Get numerical columns (excluding target)
num_types = [pl.Float32, pl.Float64, pl.Int32, pl.Int64]
num_cols = [c for c in df.columns if df.schema[c] in num_types and c != 'target']

# Calculate safe correlation: Mask -999.0 to None (null) so pl.corr ignores those rows
corr_exprs = [
    pl.corr(
        pl.when(pl.col(c) == -999.0).then(None).otherwise(pl.col(c)), 
        pl.col('target')
    ).alias(c) 
    for c in num_cols
]

# Execute the query and convert to Pandas for easy viewing
target_corrs = df.select(corr_exprs).to_pandas().T
target_corrs.columns = ['correlation_with_target']

# Sort to find the strongest positive and negative correlations
target_corrs = target_corrs.dropna().sort_values(by='correlation_with_target', key=abs, ascending=False)

top_features = target_corrs.head(15)

In [14]:
print("\n--- 3. Training Isolation Forest for Implausible Combinations ---")

# Use the top features from your previous EDA step
# If top_features is not in memory, replace this list with your best features (e.g., ['P_2', 'D_48', 'B_18', ...])
iso_features = top_features.index.tolist()

# Extract just these features to Pandas
df_iso_pd = df.select(iso_features).to_pandas()

# Train Isolation Forest on a 100k sample to save RAM
print("Fitting model on sample...")
iso_model = IsolationForest(
    n_estimators=100, 
    max_samples=100000, 
    contamination=0.01, # Assume 1% of combinations are truly implausible
    random_state=42, 
    n_jobs=-1
)
iso_model.fit(df_iso_pd.sample(n=100000, random_state=42))

# Predict on the full 5.5M rows (Returns -1 for outliers, 1 for inliers)
# We calculate decision_function to get a continuous anomaly score
print("Calculating anomaly scores for the full dataset...")
anomaly_scores = iso_model.decision_function(df_iso_pd)

# Add the new feature back to Polars
df = df.with_columns(pl.Series("anomaly_score", anomaly_scores))
print(" Anomaly detection complete. 'anomaly_score' feature added.")


--- 3. Training Isolation Forest for Implausible Combinations ---
Fitting model on sample...
Calculating anomaly scores for the full dataset...
 Anomaly detection complete. 'anomaly_score' feature added.


In [15]:
print("\n--- 4. Control Review: D_50 Outliers ---")

# Let's look at D_50 specifically
d50_control = df.select([
    pl.col('D_50').min().alias('Absolute Min (Should be -999.0)'),
    
    # Safe min/max looking ONLY at the real scaled data
    pl.when(pl.col('D_50') == -999.0).then(None).otherwise(pl.col('D_50')).min().alias('Scaled Real Min'),
    pl.when(pl.col('D_50') == -999.0).then(None).otherwise(pl.col('D_50')).max().alias('Scaled Real Max'),
])

display(d50_control.to_pandas())


--- 4. Control Review: D_50 Outliers ---


,Absolute Min (Should be -999.0),Scaled Real Min,Scaled Real Max
0,-999.0,-0.876674,7.330623


The Markers are Safe: Absolute Min is exactly -999.0. The tree models will still perfectly recognize the missing data signals.

The Outliers are Crushed: The Scaled Real Max used to be a model-breaking 244.0. It is now a mathematically stable 7.33.

The Data is Normalized: The Scaled Real Min is -0.87. This proves your Robust Scaling ((value - median) / IQR) successfully centered the real data around zero while Winsorization clipped the extreme tails.

In [16]:
print("--- Global Outlier Check ---")
# Get all numeric columns
num_cols = [c for c in df.columns if df.schema[c] in [pl.Float32, pl.Float64]]

# Find the maximum value for every column (ignoring -999.0)
exprs = [
    pl.when(pl.col(c) == -999.0).then(None).otherwise(pl.col(c)).max().alias(c)
    for c in num_cols
]

# Calculate and flip the table to sort it
max_values = df.select(exprs).to_pandas().T
max_values.columns = ['Scaled_Max_Value']

# Show the 10 highest numbers remaining in the entire dataset
display(max_values.sort_values(by='Scaled_Max_Value', ascending=False).head(10))

--- Global Outlier Check ---


,Scaled_Max_Value
D_83,203.270813
D_123,197.798691
D_140,197.276077
R_14,197.216888
R_20,197.171600
S_20,197.044891
D_93,196.915970
R_24,196.844666
R_15,196.667618
R_19,196.478989


In [18]:

# 1. Define the relative paths
save_dir = '../data/processed'
save_path = f'{save_dir}/train_data_outliers_handled.parquet'


print(f"Saving {df.estimated_size() / (1024**3):.2f} GB of data. This might take a minute...")

# 3. Write the Polars DataFrame to Parquet
df.write_parquet(save_path)

print(f" Checkpoint saved successfully at: {save_path}")

Saving 4.99 GB of data. This might take a minute...
 Checkpoint saved successfully at: ../data/processed/train_data_outliers_handled.parquet
